# LLM-Augmented Tulsa, Oklahoma Gas Price Prediction Model
### Regional Retail Gasoline Forecasting using LLM Event Sentiment & Live Pump Price Calibration ($3.89/gal)

This notebook focuses specifically on predicting **retail unleaded gasoline prices in the Tulsa, Oklahoma metropolitan area**.

### Why Tulsa, OK is Unique:
1. **Live Pump Price Calibration:** Calibrated dynamically to local pump prices (**$3.89/gal**).
2. **Cushing WTI Proximity:** Cushing, Oklahoma—the physical delivery hub for WTI crude oil—is located just **50 miles southwest of Tulsa**.
3. **Regional Refineries & Pipelines:** Local rack prices are driven by the **HF Sinclair West Tulsa Refinery** ($125,000\text{ bpd capacity}$), **Phillips 66 Ponca City Refinery**, and **Explorer Pipeline** throughput.
4. **Oklahoma State Fuel Tax:** Oklahoma maintains a low state motor fuel tax rate ($\sim \$0.19/\text{gal}$).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')

from src.tulsa_regional import fetch_tulsa_market_data, get_tulsa_regional_events
from src.event_analyzer import process_event_dataset, extract_event_features_llm
from src.feature_engineering import create_feature_matrix, prepare_chronological_splits
from src.models import train_and_compare_models

sns.set_theme(style="whitegrid")
print("Tulsa regional modules successfully loaded!")

In [ ]:
market_df = fetch_tulsa_market_data(start_date="2022-01-01", live_current_price=3.89)
raw_events_df = get_tulsa_regional_events()

print(f"Market Trading Days: {len(market_df)}")
print(f"Latest Tulsa Pump Price Calibrated: ${market_df['tulsa_retail_gasoline'].iloc[-1]:.2f}/gal")
display(market_df.head())
display(raw_events_df.head())

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(market_df['date'], market_df['tulsa_retail_gasoline'], label='Tulsa Retail Gas ($/gal)', color='tab:green', linewidth=2.5)
plt.plot(market_df['date'], market_df['gasoline_rbob'], label='Wholesale RBOB Futures ($/gal)', color='tab:blue', linestyle='--')
plt.plot(market_df['date'], market_df['cushing_crude_per_gal'], label='Cushing WTI Crude ($/gal equiv)', color='tab:orange', linestyle=':')

plt.title('Tulsa, OK Retail Gas Prices vs Wholesale RBOB & Cushing WTI Crude', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('$/Gallon')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
scored_events_df = process_event_dataset(raw_events_df, use_llm_api=False)
feature_df = create_feature_matrix(market_df, scored_events_df, forecast_horizon=5, decay_half_life_days=4.0)
splits = prepare_chronological_splits(feature_df, train_ratio=0.8, forecast_horizon=5)
results = train_and_compare_models(splits, model_type='ridge')
display(pd.DataFrame([results['metrics_quant'], results['metrics_hybrid']], index=['Baseline (Quant Only)', 'Tulsa Hybrid (Quant + LLM Events)']))